### XML → JSONL 통합 (원본 2단계 실행 순서 그대로)

1) `law_to_jsonl.ipynb` : XML → base JSONL 생성 (article 기본 is_indexable=False)
2) `add_refs_and_is_indexable.ipynb` : (a) refs 추가 → (b) is_indexable 재부여

최종적으로 `xml_to_jsonl_rag(...)` 한 번 호출하면, 위 3단계가 순차적으로 실행


In [1]:
import json
import re
import xml.etree.ElementTree as ET
from typing import Optional, Tuple

# ===== whitespace 정리 =====
_WS_STRONG = re.compile(r"\s+")
_TRAILING_SPACES = re.compile(r"[ \t]+\n")
_MULTI_BLANK_LINES = re.compile(r"\n{3,}")

# ===== 개정/주석 꼬리표: <...> 형태 =====
_ANGLE_NOTE = re.compile(r"<[^>]+>")

_CIRCLE = {
    "①": "1", "②": "2", "③": "3", "④": "4", "⑤": "5",
    "⑥": "6", "⑦": "7", "⑧": "8", "⑨": "9", "⑩": "10",
}

def norm_ws_strong(s: Optional[str]) -> str:
    """임베딩/검색용: 모든 공백을 단일 스페이스로"""
    if not s:
        return ""
    return _WS_STRONG.sub(" ", s).strip()

def norm_ws_preserve_lines(s: Optional[str]) -> str:
    """탭/불필요 공백만 정리(원문 구조 일부 유지)"""
    if not s:
        return ""
    s = s.replace("\t", " ")
    s = _TRAILING_SPACES.sub("\n", s)
    s = _MULTI_BLANK_LINES.sub("\n\n", s)
    return s.strip()

def extract_amendment_note(text: str) -> Tuple[str, Optional[str]]:
    """
    본문에서 <...> 꼬리표를 추출해 amendment_note(단일 문자열)로 저장하고,
    text에서는 제거한다.
    """
    if not text:
        return "", None

    notes = _ANGLE_NOTE.findall(text)
    cleaned = _ANGLE_NOTE.sub("", text)
    cleaned = norm_ws_strong(cleaned)

    note_str = "".join(n.strip() for n in notes if n and n.strip())
    return cleaned, (note_str if note_str else None)

def norm_no(raw: Optional[str]) -> Optional[str]:
    """항/호 번호 정규화"""
    if raw is None:
        return None
    raw = raw.strip()
    if not raw:
        return None
    raw = _CIRCLE.get(raw, raw)
    m = re.search(r"(\d+)", raw)
    return m.group(1) if m else raw

def norm_subitem_hangul(raw: Optional[str]) -> Optional[str]:
    """목번호 한글(가/나/다...) 유지"""
    if raw is None:
        return None
    r = raw.strip()
    if not r:
        return None
    m = re.search(r"(가|나|다|라|마|바|사|아|자|차|카|타|파|하)", r)
    return m.group(1) if m else r

def make_path(law_name: str, article_no: str,
              para_no: Optional[str] = None,
              item_no: Optional[str] = None,
              subitem_no: Optional[str] = None) -> str:
    parts = [law_name, f"제{article_no}조"]
    if para_no:
        parts.append(f"제{para_no}항")
    if item_no:
        parts.append(f"제{item_no}호")
    if subitem_no:
        parts.append(f"{subitem_no}목")
    return " ".join(parts)

def make_path_title(law_name: str, article_no: str, article_title: str,
                    para_no: Optional[str] = None,
                    item_no: Optional[str] = None,
                    subitem_no: Optional[str] = None) -> str:
    title_part = f"제{article_no}조"
    if article_title:
        title_part += f"({article_title})"
    parts = [law_name, title_part]
    if para_no:
        parts.append(f"제{para_no}항")
    if item_no:
        parts.append(f"제{item_no}호")
    if subitem_no:
        parts.append(f"{subitem_no}목")
    return " ".join(parts)


def xml_to_jsonl_rag_base(
    xml_path: str,
    output_path: str,
    *,
    source: str = "law_xml",
    domain: str = "statute",
    include_article_level: bool = True
) -> int:
    tree = ET.parse(xml_path)
    root = tree.getroot()

    basic = root.find("기본정보")
    if basic is None:
        raise ValueError(f"기본정보 누락: {xml_path}")

    law_id = (basic.findtext("법령ID") or "").strip()
    law_name = (basic.findtext("법령명_한글") or "").strip()
    law_type = (basic.findtext("법종구분") or "").strip()
    ministry = (basic.findtext("소관부처") or "").strip()
    promulgation_date = (basic.findtext("공포일자") or "").strip()
    enforcement_date = (basic.findtext("시행일자") or "").strip()
    revision_type = (basic.findtext("제개정구분") or "").strip()

    common = {
        "law_id": law_id,
        "law_name": law_name,
        "law_type": law_type,
        "ministry": ministry,
        "effective": {
            "promulgation_date": promulgation_date,
            "enforcement_date": enforcement_date,
            "revision_type": revision_type,
        },
        "source": source,
        "domain": domain,
    }

    def build_path(law_name_: str, article_no_display_: str, *, para_no=None, item_no=None, subitem_no=None) -> str:
        parts = [law_name_, article_no_display_]
        if para_no is not None:
            parts.append(f"제{para_no}항")
        if item_no is not None:
            parts.append(f"제{item_no}호")
        if subitem_no is not None:
            parts.append(f"{subitem_no}목")
        return " ".join(parts)

    rows = 0

    with open(output_path, "w", encoding="utf-8") as f:
        for a in root.findall(".//조문단위"):
            base_article_no = (a.findtext("조문번호") or "").strip()
            if not base_article_no:
                continue

            branch_no = (a.findtext("조문가지번호") or "").strip()
            if branch_no == "0":
                branch_no = ""

            article_id = f"{law_id}|A{base_article_no}" + (f"|{branch_no}" if branch_no else "")
            article_no_display = f"제{base_article_no}조" + (f"의{branch_no}" if branch_no else "")
            article_title = norm_ws_strong(a.findtext("조문제목", "") or "")

            # ===== article =====
            if include_article_level:
                a_src = norm_ws_preserve_lines(a.findtext("조문내용", "") or "")
                a_text, a_note = extract_amendment_note(a_src)

                if a_text:
                    rec_a = {
                        **common,
                        "doc_id": article_id,
                        "parent_id": None,
                        "level": "article",
                        "is_indexable": False,
                        "article_no": article_no_display,
                        "article_title": article_title,
                        "paragraph_no": None,
                        "item_no": None,
                        "subitem_no": None,
                        "path": build_path(law_name, article_no_display),
                        "text": a_text,
                        "amendment_note": a_note,
                    }
                    f.write(json.dumps(rec_a, ensure_ascii=False) + "\n")
                    rows += 1

            # 조문단위 바로 아래 '호', '항', '목'
            direct_items = a.findall("호")
            clauses = a.findall("항")
            direct_subitems = a.findall("목")  # ✅ NEW: 조 → 목 직접 케이스

            # ===== CASE 0) 항/호 없이 조 -> 목 로 바로 가는 경우 =====
            if not clauses and not direct_items and direct_subitems:
                for sub in direct_subitems:
                    sub_no = norm_subitem_hangul(sub.findtext("목번호"))
                    if not sub_no:
                        continue

                    sub_id = f"{article_id}|M{sub_no}"  # ✅ NEW: 조 직하 목은 A..(|가지)|M가
                    sub_src = norm_ws_preserve_lines(sub.findtext("목내용", "") or "")
                    sub_text, sub_note = extract_amendment_note(sub_src)
                    if not sub_text:
                        continue

                    rec_m = {
                        **common,
                        "doc_id": sub_id,
                        "parent_id": article_id,     # ✅ 조에 직접 연결
                        "level": "subitem",
                        "is_indexable": True,
                        "article_no": article_no_display,
                        "article_title": article_title,
                        "paragraph_no": None,
                        "item_no": None,
                        "subitem_no": sub_no,
                        "path": build_path(law_name, article_no_display, subitem_no=sub_no),
                        "text": sub_text,
                        "amendment_note": sub_note,
                    }
                    f.write(json.dumps(rec_m, ensure_ascii=False) + "\n")
                    rows += 1

                continue  # 이 조문단위 처리 종료

            # ===== CASE 1) 항이 없고, 조 -> 호 로 바로 가는 경우 =====
            if not clauses and direct_items:
                for it in direct_items:
                    item_no = norm_no(it.findtext("호번호"))
                    item_id = f"{article_id}|I{item_no or 'X'}"

                    it_src = norm_ws_preserve_lines(it.findtext("호내용", "") or "")
                    it_text, it_note = extract_amendment_note(it_src)

                    subs = it.findall("목")

                    if it_text or subs:
                        rec_i = {
                            **common,
                            "doc_id": item_id,
                            "parent_id": article_id,
                            "level": "item",
                            "is_indexable": True,
                            "article_no": article_no_display,
                            "article_title": article_title,
                            "paragraph_no": None,
                            "item_no": item_no,
                            "subitem_no": None,
                            "path": build_path(law_name, article_no_display, item_no=item_no),
                            "text": it_text or "",
                            "amendment_note": it_note,
                        }
                        f.write(json.dumps(rec_i, ensure_ascii=False) + "\n")
                        rows += 1

                    for sub in subs:
                        sub_no = norm_subitem_hangul(sub.findtext("목번호"))
                        sub_id = f"{item_id}|M{sub_no or 'X'}"

                        sub_src = norm_ws_preserve_lines(sub.findtext("목내용", "") or "")
                        sub_text, sub_note = extract_amendment_note(sub_src)
                        if not sub_text:
                            continue

                        rec_m = {
                            **common,
                            "doc_id": sub_id,
                            "parent_id": item_id,
                            "level": "subitem",
                            "is_indexable": True,
                            "article_no": article_no_display,
                            "article_title": article_title,
                            "paragraph_no": None,
                            "item_no": item_no,
                            "subitem_no": sub_no,
                            "path": build_path(law_name, article_no_display, item_no=item_no, subitem_no=sub_no),
                            "text": sub_text,
                            "amendment_note": sub_note,
                        }
                        f.write(json.dumps(rec_m, ensure_ascii=False) + "\n")
                        rows += 1

                continue

            # ===== CASE 2) 항이 있는 일반 구조 =====
            if not clauses:
                continue

            for p in clauses:
                para_no = norm_no(p.findtext("항번호"))
                para_id = f"{article_id}|P{para_no or 'X'}"

                p_src = norm_ws_preserve_lines(p.findtext("항내용", "") or "")
                p_text, p_note = extract_amendment_note(p_src)

                items = p.findall("호")
                para_subitems = p.findall("목")  # ✅ NEW: 항 → 목 직접 케이스

                # ✅ PX 껍데기 판단에 '목'도 포함
                is_shell_px = (para_no is None) and (not p_text) and (bool(items) or bool(para_subitems))

                # ✅ paragraph 저장 조건에도 '목' 포함
                should_write_paragraph = bool(p_text) or (para_no is not None and (bool(items) or bool(para_subitems)))

                if should_write_paragraph and not is_shell_px:
                    rec_p = {
                        **common,
                        "doc_id": para_id,
                        "parent_id": article_id,
                        "level": "paragraph",
                        "is_indexable": True,
                        "article_no": article_no_display,
                        "article_title": article_title,
                        "paragraph_no": para_no,
                        "item_no": None,
                        "subitem_no": None,
                        "path": build_path(law_name, article_no_display, para_no=para_no),
                        "text": p_text or "",
                        "amendment_note": p_note,
                    }
                    f.write(json.dumps(rec_p, ensure_ascii=False) + "\n")
                    rows += 1

                # 항 직하 목의 parent:
                # - PX 껍데기면 article
                # - 아니면 paragraph
                subitem_parent_id = article_id if is_shell_px else para_id

                # ===== CASE 2-0) 조 -> 항 -> 목 (호 없이) =====
                if para_subitems:
                    for sub in para_subitems:
                        sub_no = norm_subitem_hangul(sub.findtext("목번호"))
                        if not sub_no:
                            continue

                        # doc_id 규칙:
                        # - PX 껍데기면 A..(|가지)|M가
                        # - 일반 항이면 A..(|가지)|P..|M가
                        if is_shell_px:
                            sub_id = f"{article_id}|M{sub_no}"
                            para_no_for_path = None
                        else:
                            sub_id = f"{article_id}|P{para_no or 'X'}|M{sub_no}"
                            para_no_for_path = para_no

                        sub_src = norm_ws_preserve_lines(sub.findtext("목내용", "") or "")
                        sub_text, sub_note = extract_amendment_note(sub_src)
                        if not sub_text:
                            continue

                        rec_m = {
                            **common,
                            "doc_id": sub_id,
                            "parent_id": subitem_parent_id,
                            "level": "subitem",
                            "is_indexable": True,
                            "article_no": article_no_display,
                            "article_title": article_title,
                            "paragraph_no": para_no_for_path,
                            "item_no": None,
                            "subitem_no": sub_no,
                            "path": build_path(law_name, article_no_display, para_no=para_no_for_path, subitem_no=sub_no),
                            "text": sub_text,
                            "amendment_note": sub_note,
                        }
                        f.write(json.dumps(rec_m, ensure_ascii=False) + "\n")
                        rows += 1

                # item의 parent 결정(기존 로직 유지)
                item_parent_id = article_id if is_shell_px else para_id

                for it in items:
                    item_no = norm_no(it.findtext("호번호"))

                    if is_shell_px:
                        item_id = f"{article_id}|I{item_no or 'X'}"
                    else:
                        item_id = f"{article_id}|P{para_no or 'X'}|I{item_no or 'X'}"

                    it_src = norm_ws_preserve_lines(it.findtext("호내용", "") or "")
                    it_text, it_note = extract_amendment_note(it_src)

                    subs = it.findall("목")

                    if it_text or subs:
                        rec_i = {
                            **common,
                            "doc_id": item_id,
                            "parent_id": item_parent_id,
                            "level": "item",
                            "is_indexable": True,
                            "article_no": article_no_display,
                            "article_title": article_title,
                            "paragraph_no": None if is_shell_px else para_no,
                            "item_no": item_no,
                            "subitem_no": None,
                            "path": build_path(
                                law_name,
                                article_no_display,
                                para_no=None if is_shell_px else para_no,
                                item_no=item_no
                            ),
                            "text": it_text or "",
                            "amendment_note": it_note,
                        }
                        f.write(json.dumps(rec_i, ensure_ascii=False) + "\n")
                        rows += 1

                    for sub in subs:
                        sub_no = norm_subitem_hangul(sub.findtext("목번호"))
                        sub_id = f"{item_id}|M{sub_no or 'X'}"

                        sub_src = norm_ws_preserve_lines(sub.findtext("목내용", "") or "")
                        sub_text, sub_note = extract_amendment_note(sub_src)
                        if not sub_text:
                            continue

                        rec_m = {
                            **common,
                            "doc_id": sub_id,
                            "parent_id": item_id,
                            "level": "subitem",
                            "is_indexable": True,
                            "article_no": article_no_display,
                            "article_title": article_title,
                            "paragraph_no": None if is_shell_px else para_no,
                            "item_no": item_no,
                            "subitem_no": sub_no,
                            "path": build_path(
                                law_name,
                                article_no_display,
                                para_no=None if is_shell_px else para_no,
                                item_no=item_no,
                                subitem_no=sub_no
                            ),
                            "text": sub_text,
                            "amendment_note": sub_note,
                        }
                        f.write(json.dumps(rec_m, ensure_ascii=False) + "\n")
                        rows += 1

    print(f"✅ {rows} rows saved to {output_path}")
    return rows


In [30]:
import re
from typing import List, Dict, Any, Tuple, Optional

# =====================================================
# 1. 외부 법령 블록
# =====================================================
LAW_BLOCK_RE = re.compile(r"「(?P<law>[^」]+)」(?P<body>[^「]*)")

# =====================================================
# 2. 조문 패턴 (완전판)
# =====================================================
# ARTICLE_RANGE_RE = re.compile(
#     r"제(?P<s>\d+)조(?:의(?P<sd>\d+))?\s*부터\s*제(?P<e>\d+)조(?:의(?P<ed>\d+))?\s*까지"
# )

ARTICLE_RANGE_RE = re.compile(
    r"제(?P<s>\d+)조(?:의(?P<sd>\d+))?\s*(?:부터|내지)\s*제(?P<e>\d+)조(?:의(?P<ed>\d+))?"
)


PARA_ITEM_RE = re.compile(
    r"제(?P<a>\d+)조(?:의(?P<ad>\d+))?\s*제(?P<p>\d+)항\s*제(?P<i>\d+)호"
)

PARA_RE = re.compile(
    r"제(?P<a>\d+)조(?:의(?P<ad>\d+))?\s*제(?P<p>\d+)항"
)

ARTICLE_ITEM_RE = re.compile(
    r"제(?P<a>\d+)조(?:의(?P<ad>\d+))?\s*제(?P<i>\d+)호"
)

PARA_ONLY_RE = re.compile(r"제(?P<p>\d+)항")
ITEM_RANGE_RE = re.compile(r"제(?P<s>\d+)호\s*부터\s*제(?P<e>\d+)호\s*까지")
ITEM_ONLY_RE = re.compile(r"제(?P<i>\d+)호")

SPLIT_SEP_RE = re.compile(r"\s*(?:,|·|및|와|과)\s*")

# =====================================================
# 3. body 전체 스캔 파서 (🔥 핵심 수정)
# =====================================================
def parse_refs_from_body(body: str) -> List[Dict[str, Any]]:
    refs: List[Dict[str, Any]] = []

    # 1) 조 범위
    for m in ARTICLE_RANGE_RE.finditer(body):
        refs.append({
            "type": "article_range",
            "start": m.group("s"),
            "end": m.group("e"),
        })

    # 2) 조-항-호
    for m in PARA_ITEM_RE.finditer(body):
        refs.append({
            "type": "article_paragraph_item",
            "article": m.group("a"),
            "paragraph": int(m.group("p")),
            "item": int(m.group("i")),
        })

    # 3) 조-항 + 항 나열 묶기 (🔥 수정된 부분)
    para_bucket: Dict[str, set] = {}
    last_article: Optional[str] = None

    tokens = [t.strip() for t in SPLIT_SEP_RE.split(body) if t.strip()]

    for t in tokens:
        m_full = PARA_RE.search(t)
        if m_full:
            article = m_full.group("a")
            para = int(m_full.group("p"))
            last_article = article
            para_bucket.setdefault(article, set()).add(para)
            continue

        m_only = PARA_ONLY_RE.search(t)
        if m_only and last_article:
            para_bucket.setdefault(last_article, set()).add(int(m_only.group("p")))
            continue

    for article, paras in para_bucket.items():
        refs.append({
            "type": "article_paragraphs",
            "article": article,
            "paragraphs": sorted(paras),
        })

    # 4) 조-호
    for m in ARTICLE_ITEM_RE.finditer(body):
        refs.append({
            "type": "article_item",
            "article": m.group("a"),
            "item": int(m.group("i")),
        })

    # 5) 호 범위
    for m in ITEM_RANGE_RE.finditer(body):
        refs.append({
            "type": "item_range",
            "start": int(m.group("s")),
            "end": int(m.group("e")),
        })

    return refs


# =====================================================
# 4. 메인 함수
# =====================================================
def extract_law_refs_and_mentions(
    text: str,
    law_name_to_id: Dict[str, str]
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:

    ref_citations: List[Dict[str, Any]] = []
    mentioned_laws: List[Dict[str, Any]] = []

    for m in LAW_BLOCK_RE.finditer(text):
        law_name = m.group("law").strip()
        body = m.group("body").strip()
        law_id = law_name_to_id.get(law_name)

        refs = parse_refs_from_body(body)

        if refs:
            ref_citations.append({
                "law_name": law_name,
                "law_id": law_id,
                "raw": f"「{law_name}」{body}",
                "refs": refs,
            })
        else:
            mentioned_laws.append({
                "law_name": law_name,
                "law_id": law_id,
                "raw": f"「{law_name}」",
            })

    return ref_citations, mentioned_laws


# =====================================================
# 5. 실행 테스트
# =====================================================


import re
from typing import List, Dict, Any, Optional

# ===============================
# 내부 법령 조문 패턴
# ===============================

ARTICLE_RE = re.compile(
    r"제(?P<a>\d+)조(?:의(?P<ad>\d+))?"
)

ARTICLE_RANGE_RE = re.compile(
    r"제(?P<s>\d+)조(?:의(?P<sd>\d+))?\s*(?:부터|내지)\s*제(?P<e>\d+)조(?:의(?P<ed>\d+))?"
)

PARA_RE = re.compile(
    r"제(?P<a>\d+)조(?:의(?P<ad>\d+))?\s*제(?P<p>\d+)항"
)

PARA_ONLY_RE = re.compile(r"제(?P<p>\d+)항")

PARA_RANGE_RE = re.compile(
    r"제(?P<s>\d+)항\s*부터\s*제(?P<e>\d+)항\s*까지"
)

ITEM_RE = re.compile(
    r"제(?P<a>\d+)조(?:의(?P<ad>\d+))?\s*제(?P<p>\d+)항\s*제(?P<i>\d+)호"
)

ITEM_ONLY_RE = re.compile(r"제(?P<i>\d+)호")

SPLIT_RE = re.compile(r"[ㆍ·,]|및")


# ===============================
# 내부 법령 파서
# ===============================
def parse_internal_refs(text: str) -> List[Dict[str, Any]]:
    """
    같은 법령 내부 조문 인용을 구조화하여 추출한다.
    """
    refs: List[Dict[str, Any]] = []

    # 1️⃣ 조 범위 (제402조 내지 제407조)
    for m in ARTICLE_RANGE_RE.finditer(text):
        refs.append({
            "type": "article_range",
            "start": f"{m.group('s')}{('의'+m.group('sd')) if m.group('sd') else ''}",
            "end": f"{m.group('e')}{('의'+m.group('ed')) if m.group('ed') else ''}",
        })

    # 2️⃣ 단일 조
    for m in ARTICLE_RE.finditer(text):
        refs.append({
            "type": "article",
            "article": f"{m.group('a')}{('의'+m.group('ad')) if m.group('ad') else ''}",
        })

    # 3️⃣ 조-항 / 항 나열 / 항 범위
    para_bucket: Dict[str, set] = {}
    last_article: Optional[str] = None

    tokens = [t.strip() for t in SPLIT_RE.split(text) if t.strip()]

    for t in tokens:
        # 조-항
        m_full = PARA_RE.search(t)
        if m_full:
            article = f"{m_full.group('a')}{('의'+m_full.group('ad')) if m_full.group('ad') else ''}"
            para = int(m_full.group("p"))
            last_article = article
            para_bucket.setdefault(article, set()).add(para)
            continue

        # 항 범위 (제1항부터 제3항까지)
        m_pr = PARA_RANGE_RE.search(t)
        if m_pr and last_article:
            for p in range(int(m_pr.group("s")), int(m_pr.group("e")) + 1):
                para_bucket.setdefault(last_article, set()).add(p)
            continue

        # 단독 항 (제4항)
        m_only = PARA_ONLY_RE.search(t)
        if m_only and last_article:
            para_bucket.setdefault(last_article, set()).add(int(m_only.group("p")))
            continue

    for article, paras in para_bucket.items():
        refs.append({
            "type": "article_paragraphs",
            "article": article,
            "paragraphs": sorted(paras),
        })

    # 4️⃣ 조-항-호 / 호 나열
    item_bucket: Dict[str, Dict[int, set]] = {}
    last_article_item: Optional[str] = None
    last_para_item: Optional[int] = None

    for t in tokens:
        # 조-항-호
        m_item = ITEM_RE.search(t)
        if m_item:
            article = f"{m_item.group('a')}{('의'+m_item.group('ad')) if m_item.group('ad') else ''}"
            para = int(m_item.group("p"))
            item = int(m_item.group("i"))

            last_article_item = article
            last_para_item = para

            item_bucket.setdefault(article, {}).setdefault(para, set()).add(item)
            continue

        # 단독 호 (제9호)
        m_only_item = ITEM_ONLY_RE.search(t)
        if m_only_item and last_article_item and last_para_item:
            item_bucket[last_article_item][last_para_item].add(int(m_only_item.group("i")))

    for article, paras in item_bucket.items():
        for para, items in paras.items():
            refs.append({
                "type": "article_paragraph_items",
                "article": article,
                "paragraph": para,
                "items": sorted(items),
            })

    return refs


import re
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict

# ============================================================
# 0) 마커(문맥 전환) 패턴: 외부 법령명, 같은 법, 이 법
# ============================================================
MARKER_RE = re.compile(
    r"(?P<ext>「(?P<ext_name>[^」]+)」)|(?P<same>같은 법)|(?P<this>이 법)"
)

# ============================================================
# 1) 공통 조문 패턴들
# ============================================================
ARTICLE_SINGLE_RE = re.compile(r"제(?P<a>\d+)조(?:의(?P<ad>\d+))?")
ARTICLE_PARA_RE = re.compile(r"제(?P<a>\d+)조(?:의(?P<ad>\d+))?\s*제(?P<p>\d+)항")
ARTICLE_PARA_ITEM_RE = re.compile(r"제(?P<a>\d+)조(?:의(?P<ad>\d+))?\s*제(?P<p>\d+)항\s*제(?P<i>\d+)호")
ARTICLE_ITEM_RE = re.compile(r"제(?P<a>\d+)조(?:의(?P<ad>\d+))?\s*제(?P<i>\d+)호")

ARTICLE_RANGE_RE = re.compile(
    r"제(?P<s>\d+)조(?:의(?P<sd>\d+))?\s*(?:부터|내지)\s*제(?P<e>\d+)조(?:의(?P<ed>\d+))?\s*(?:까지)?"
)
PARA_RANGE_ONLY_RE = re.compile(r"제(?P<s>\d+)항\s*부터\s*제(?P<e>\d+)항\s*까지")
ITEM_RANGE_ONLY_RE = re.compile(r"제(?P<s>\d+)호\s*부터\s*제(?P<e>\d+)호\s*까지")

PARA_ONLY_RE = re.compile(r"제(?P<p>\d+)항")
ITEM_ONLY_RE = re.compile(r"제(?P<i>\d+)호")

SPLIT_SEP_RE = re.compile(r"\s*(?:,|ㆍ|·|및|와|과)\s*")


def _article_key(a: str, ad: Optional[str]) -> str:
    return f"{a}{('의' + ad) if ad else ''}"


# ============================================================
# 2) 한 span(마커~다음 마커 사이)에서 refs를 전부 추출
#    ✅ 수정: "제2조제1호 및 제2호"에서 제2호도 같은 조-호로 붙이기
# ============================================================
def extract_refs_from_span(span: str) -> List[Dict[str, Any]]:
    refs: List[Dict[str, Any]] = []

    # (1) 조 범위 먼저 수집
    for m in ARTICLE_RANGE_RE.finditer(span):
        refs.append({
            "type": "article_range",
            "start": _article_key(m.group("s"), m.group("sd")),
            "end": _article_key(m.group("e"), m.group("ed")),
        })

    tokens = [t.strip() for t in SPLIT_SEP_RE.split(span) if t.strip()]

    last_article: Optional[str] = None
    last_para: Optional[int] = None

    # ✅ 새로 추가: "조-호(항 없음)" 컨텍스트용
    last_article_for_items_no_para: Optional[str] = None

    # 그룹 버킷
    para_bucket: Dict[str, set] = defaultdict(set)                 # article -> {paras}
    item_bucket_with_para: Dict[Tuple[str, int], set] = defaultdict(set)  # (article, para) -> {items}
    item_bucket_no_para: Dict[str, set] = defaultdict(set)         # article -> {items}  (항 없음)

    for t in tokens:
        # 1) 조-항-호
        m = ARTICLE_PARA_ITEM_RE.search(t)
        if m:
            akey = _article_key(m.group("a"), m.group("ad"))
            p = int(m.group("p"))
            i = int(m.group("i"))
            refs.append({
                "type": "article_paragraph_item",
                "article": akey,
                "paragraph": p,
                "item": i,
            })
            last_article, last_para = akey, p
            last_article_for_items_no_para = None  # 항 있는 컨텍스트로 전환
            item_bucket_with_para[(akey, p)].add(i)
            continue

        # 2) 조-항
        m = ARTICLE_PARA_RE.search(t)
        if m:
            akey = _article_key(m.group("a"), m.group("ad"))
            p = int(m.group("p"))
            last_article, last_para = akey, p
            last_article_for_items_no_para = None
            para_bucket[akey].add(p)
            continue

        # 3) 조-호(항 없음)  ✅ 여기서 컨텍스트 기억
        m = ARTICLE_ITEM_RE.search(t)
        if m:
            akey = _article_key(m.group("a"), m.group("ad"))
            i = int(m.group("i"))
            refs.append({
                "type": "article_item",
                "article": akey,
                "item": i,
            })
            last_article, last_para = akey, None
            last_article_for_items_no_para = akey  # ✅ 이후 "제2호"를 여기로 붙일 수 있게
            item_bucket_no_para[akey].add(i)
            continue

        # 4) 단일 조
        m = ARTICLE_SINGLE_RE.search(t)
        if m:
            akey = _article_key(m.group("a"), m.group("ad"))
            refs.append({
                "type": "article",
                "article": akey,
            })
            last_article, last_para = akey, None
            last_article_for_items_no_para = None
            continue

        # 5) 항 범위(조 생략) -> 직전 article 필요
        m = PARA_RANGE_ONLY_RE.search(t)
        if m and last_article:
            s, e = int(m.group("s")), int(m.group("e"))
            for p in range(s, e + 1):
                para_bucket[last_article].add(p)
            last_para = e
            last_article_for_items_no_para = None
            continue

        # 6) 단독 항 -> 직전 article 필요
        m = PARA_ONLY_RE.search(t)
        if m and last_article:
            p = int(m.group("p"))
            para_bucket[last_article].add(p)
            last_para = p
            last_article_for_items_no_para = None
            continue

        # 7) 호 범위(조/항 생략)
        m = ITEM_RANGE_ONLY_RE.search(t)
        if m:
            s, e = int(m.group("s")), int(m.group("e"))
            # (a) 조-항 컨텍스트면 (article, para)에 붙임
            if last_article and last_para is not None:
                for i in range(s, e + 1):
                    item_bucket_with_para[(last_article, last_para)].add(i)
                continue
            # (b) ✅ 조-호(항 없음) 컨텍스트면 article에 붙임
            if last_article_for_items_no_para:
                for i in range(s, e + 1):
                    item_bucket_no_para[last_article_for_items_no_para].add(i)
                continue

        # 8) 단독 호
        m = ITEM_ONLY_RE.search(t)
        if m:
            i = int(m.group("i"))
            # (a) 조-항 컨텍스트
            if last_article and last_para is not None:
                item_bucket_with_para[(last_article, last_para)].add(i)
                continue
            # (b) ✅ 조-호(항 없음) 컨텍스트
            if last_article_for_items_no_para:
                item_bucket_no_para[last_article_for_items_no_para].add(i)
                continue

    # 그룹 버킷 -> refs로 출력
    for akey, ps in para_bucket.items():
        if ps:
            refs.append({
                "type": "article_paragraphs",
                "article": akey,
                "paragraphs": sorted(ps),
            })

    for (akey, p), items in item_bucket_with_para.items():
        if items:
            refs.append({
                "type": "article_paragraph_items",
                "article": akey,
                "paragraph": p,
                "items": sorted(items),
            })

    # ✅ 항 없는 조-호 그룹도 출력
    for akey, items in item_bucket_no_para.items():
        if items:
            refs.append({
                "type": "article_items",
                "article": akey,
                "items": sorted(items),
            })

    return refs


# ============================================================
# 3) 메인 파서: 내부/외부 한 번에
# ============================================================
def parse_internal_external_citations(
    *,
    text: str,
    this_law_id: str,
    this_law_name: str,
    law_name_to_id: Dict[str, str],
) -> Dict[str, Any]:

    refs_map: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    mentioned_set = set()

    active_law_key = "__THIS__"
    last_external_law_name: Optional[str] = None

    last_pos = 0
    for m in MARKER_RE.finditer(text):
        span = text[last_pos:m.start()]
        span_refs = extract_refs_from_span(span)
        if span_refs:
            refs_map[active_law_key].extend(span_refs)

        if m.group("ext"):
            ext_name = (m.group("ext_name") or "").strip()
            last_external_law_name = ext_name
            active_law_key = ext_name
            mentioned_set.add(ext_name)

        elif m.group("same"):
            if last_external_law_name:
                active_law_key = last_external_law_name
                mentioned_set.add(last_external_law_name)

        elif m.group("this"):
            active_law_key = "__THIS__"

        last_pos = m.end()

    tail = text[last_pos:]
    tail_refs = extract_refs_from_span(tail)
    if tail_refs:
        refs_map[active_law_key].extend(tail_refs)

    internal_refs = refs_map.get("__THIS__", [])

    external_list = []
    for law_name, refs in refs_map.items():
        if law_name == "__THIS__":
            continue
        if not refs:
            continue
        external_list.append({
            "law_name": law_name,
            "law_id": law_name_to_id.get(law_name),
            "refs": refs,
        })

    mentioned_laws = []
    for law_name in sorted(mentioned_set):
        has_refs = bool(refs_map.get(law_name))
        if not has_refs:
            mentioned_laws.append({
                "law_name": law_name,
                "law_id": law_name_to_id.get(law_name),
            })

    return {
        "ref_citations_internal": [{
            "law_name": this_law_name,
            "law_id": this_law_id,
            "refs": internal_refs,
        }] if internal_refs else [],
        "ref_citations_external": external_list,
        "mentioned_laws": mentioned_laws,
    }


# ============================================================
# 4) 테스트
# ============================================================


from typing import Dict, Any, List, Optional

def _article_key_to_article_no(article_key: str) -> str:
    """
    parser가 만든 article 키: "449" 또는 "449의2"
    -> jsonl 스타일 article_no: "제449조" 또는 "제449조의2"
    """
    if "의" in article_key:
        base, sub = article_key.split("의", 1)
        return f"제{base}조의{sub}"
    return f"제{article_key}조"

def _paragraph_no(p: Optional[int]) -> Optional[str]:
    return str(p) if p is not None else None

def _item_no(i: Optional[int]) -> Optional[str]:
    return str(i) if i is not None else None

def normalize_refs_for_jsonl(
    parsed: Dict[str, Any],
    *,
    this_law_id: str,
    this_law_name: str,
) -> Dict[str, Any]:
    """
    parse_internal_external_citations(...) 결과를
    jsonl(laws_nodes) 필드와 맞는 구조로 변환한다.

    반환:
      {
        "ref_internal": [ ... ],
        "ref_external": [ ... ],
        "mentioned_laws": [ ... ]   # 기존 그대로 (필요하면 유지)
      }
    """

    def convert_ref_list(refs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        out: List[Dict[str, Any]] = []

        for r in refs:
            t = r.get("type")

            # 1) article_range: start/end가 "402" / "449의2" 형태
            if t == "article_range":
                out.append({
                    "type": "article_range",
                    "start_article_no": _article_key_to_article_no(r["start"]),
                    "end_article_no": _article_key_to_article_no(r["end"]),
                    "start_paragraph_no": None,
                    "end_paragraph_no": None,
                    "start_item_no": None,
                    "end_item_no": None,
                })
                continue

            # 2) article
            if t == "article":
                out.append({
                    "type": "article",
                    "article_no": _article_key_to_article_no(r["article"]),
                    "paragraph_no": None,
                    "item_no": None,
                })
                continue

            # 3) article_paragraph_item (단건)
            if t == "article_paragraph_item":
                out.append({
                    "type": "citation",
                    "article_no": _article_key_to_article_no(r["article"]),
                    "paragraph_no": _paragraph_no(r["paragraph"]),
                    "item_no": _item_no(r["item"]),
                })
                continue

            # 4) article_paragraph (단건)
            if t == "article_paragraph":
                out.append({
                    "type": "citation",
                    "article_no": _article_key_to_article_no(r["article"]),
                    "paragraph_no": _paragraph_no(r["paragraph"]),
                    "item_no": None,
                })
                continue

            # 5) article_paragraphs (묶음) : article + paragraphs[]
            if t == "article_paragraphs":
                out.append({
                    "type": "paragraph_list",
                    "article_no": _article_key_to_article_no(r["article"]),
                    "paragraph_nos": [str(p) for p in r.get("paragraphs", [])],
                })
                continue

            # 6) article_paragraph_items (묶음): article + paragraph + items[]
            if t == "article_paragraph_items":
                out.append({
                    "type": "item_list",
                    "article_no": _article_key_to_article_no(r["article"]),
                    "paragraph_no": _paragraph_no(r["paragraph"]),
                    "item_nos": [str(i) for i in r.get("items", [])],
                })
                continue

            # 7) article_item (단건): 항 없이 조-호
            if t == "article_item":
                out.append({
                    "type": "citation",
                    "article_no": _article_key_to_article_no(r["article"]),
                    "paragraph_no": None,
                    "item_no": _item_no(r["item"]),
                })
                continue

            # 8) article_items (묶음): 항 없이 조-호 나열
            if t == "article_items":
                out.append({
                    "type": "item_list",
                    "article_no": _article_key_to_article_no(r["article"]),
                    "paragraph_no": None,
                    "item_nos": [str(i) for i in r.get("items", [])],
                })
                continue

            # 9) item_range / paragraph_range 등(혹시 남는 타입 대비)
            #    현재 너의 최신 파서는 range를 bucket에만 담고 refs에는 안 남길 수 있는데,
            #    남아있으면 최소한 구조를 유지해둔다.
            if t in ("item_range", "paragraph_range"):
                out.append(r)
                continue

            # 알 수 없는 타입은 그대로 보존 (디버깅용)
            out.append(r)

        return out

    # ----------------------------
    # 내부/외부 refs 꺼내기
    # ----------------------------
    internal = []
    ext = []

    # parsed["ref_citations_internal"] 는 리스트(없으면 [])
    if parsed.get("ref_citations_internal"):
        # 내부는 항상 현재 law를 의미 (구조 통일)
        internal_refs = parsed["ref_citations_internal"][0].get("refs", [])
        internal = [{
            "law_id": this_law_id,
            "law_name": this_law_name,
            "refs": convert_ref_list(internal_refs),
        }]

    # 외부
    for e in parsed.get("ref_citations_external", []):
        ext.append({
            "law_id": e.get("law_id"),
            "law_name": e.get("law_name"),
            "refs": convert_ref_list(e.get("refs", [])),
        })

    return {
        "ref_internal": internal,                 # jsonl에 붙일 필드
        "ref_external": ext,                      # jsonl에 붙일 필드
        "mentioned_laws": parsed.get("mentioned_laws", []),  # 필요하면 유지
    }


import json
from typing import Dict


def load_law_name_map(law_map_path: str) -> Dict[str, str]:
    """
    법령과 법령ID 정보 있는 파일을 읽어
    {법령명 or 법령약칭 -> law_id} dict 만듦.
    """
    name_to_id: Dict[str, str] = {}

    with open(law_map_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue

            law_id = row.get("law_id")
            if not law_id:
                continue

            # 정식 법령명
            law_name = row.get("law_name")
            if law_name:
                name_to_id[law_name.strip()] = law_id

            # 약칭도 같이 매핑
            law_short = row.get("law_short_name")
            if law_short:
                name_to_id[law_short.strip()] = law_id

    return name_to_id


import json
from typing import Dict, Any
import re



def enrich_jsonl_with_refs(
    input_path: str,
    output_path: str,
    law_name_to_id: Dict[str, str],
) -> int:
    n = 0

    with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
        for line_no, line in enumerate(fin, 1):
            line = line.strip()
            if not line:
                continue

            try:
                row: Dict[str, Any] = json.loads(line)
            except json.JSONDecodeError:
                continue


            # 조문 머리말 제거: 맨 앞의 "제N조(....)" 또는 "제N조의M(....)" 패턴
            _HEAD_ARTICLE_TITLE_RE = re.compile(r"^\s*제\d+조(?:의\d+)?\([^)]*\)\s*")

            text = row.get("text", "") or ""
            this_law_id = row.get("law_id", "") or ""
            this_law_name = row.get("law_name", "") or ""

            # 파서에 넣을 텍스트만 별도로 만듦 (원본 text는 유지)
            text_for_parse = text

            # 보수적으로: row가 article 레벨이거나, text가 진짜 조문머리말로 시작할 때만 제거
            # (원하면 level 조건은 빼도 됨. ^앵커라서 부작용 거의 없음)
            if _HEAD_ARTICLE_TITLE_RE.match(text_for_parse):
                text_for_parse = _HEAD_ARTICLE_TITLE_RE.sub("", text_for_parse, count=1)

            parsed = parse_internal_external_citations(
                text=text_for_parse,            # ✅ 여기만 바뀜
                this_law_id=this_law_id,
                this_law_name=this_law_name,
                law_name_to_id=law_name_to_id
            )


            # 결과 필드 추가
            row["ref_citations_internal"] = parsed.get("ref_citations_internal", [])
            row["ref_citations_external"] = parsed.get("ref_citations_external", [])
            row["mentioned_laws"] = parsed.get("mentioned_laws", [])
            
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
            n += 1

    return n

law_name_to_id = load_law_name_map('../data/law_map.jsonl')


import json
import re
from typing import Dict, Any, List, Optional

# =========================================================
# 1) 규칙 1: article 중 "제n편/장/절/관" 표제는 False
# =========================================================
HEADING_RE = re.compile(r"^\s*제\s*\d+\s*(편|장|절|관)\b")

def is_heading_article_text(text: str) -> bool:
    return bool(HEADING_RE.match((text or "").strip()))


# =========================================================
# 2) 규칙 2: 링크전용 준용이면 False
# =========================================================
REF_PAT = re.compile(r"제\d+조(?:의\d+)?(?:제\d+항)?(?:제\d+호)?")
SUB_REF_PAT = re.compile(r"제\d+항|제\d+호|제\d+목")  # 필요하면 목도
JUNYONG_PAT = re.compile(r"준용")

STOPWORDS = [
    "규정", "경우", "따라", "따른다", "적용", "준용", "한다", "하여", "이를", "및", "또는",
    "의", "에", "을", "를", "내지",
    "전항", "전조", "전호", "각호", "전단", "후단", "이항", "이조", "본조", "본항"
]

def strip_refs(text: str) -> str:
    t = REF_PAT.sub(" ", text)
    t = SUB_REF_PAT.sub(" ", t)  # 조 없이 등장하는 항/호/목도 제거
    t = re.sub(r"[0-9①②③④⑤⑥⑦⑧⑨⑩\(\)\[\]·,\.]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def meaningful_length(text_wo_refs: str) -> int:
    t = text_wo_refs
    for w in STOPWORDS:
        t = t.replace(w, " ")
    t = re.sub(r"\s+", "", t)
    return len(t)

def is_link_only_junyong(text: str) -> bool:
    if not JUNYONG_PAT.search(text):
        return False
    if not REF_PAT.search(text):
        return False

    wo = strip_refs(text)
    ml = meaningful_length(wo)
    return ml <= 5


# =========================================================
# 2) 규칙 2-1: text 내용이 "제n조(준용규정)"이면 False
# =========================================================
JUNYONG_HEADER_ONLY_PAT = re.compile(
    r"^\s*제\s*\d+\s*조(?:\s*의\s*\d+\s*)?\s*\(\s*준용규정\s*\)\s*$"
)

def is_junyong_header_only(article_title: str, text: str) -> bool:
    """
    article_title이 '준용규정'이고, text가 '제n조(준용규정)' 형태로
    헤더만 있는 경우 True.
    """
    if (article_title or "").strip() != "준용규정":
        return False
    t = (text or "").strip()
    return bool(JUNYONG_HEADER_ONLY_PAT.fullmatch(t))


# =========================================================
# 2) 규칙 3: '제nn조 삭제'라고 써져있는 조문이면 False
# =========================================================

# 조 삭제: "제436조 삭제", "제436조의2 삭제" 같은 것까지 포함
DEL_ARTICLE_PAT = re.compile(r"^\s*제\s*\d+\s*조(?:\s*의\s*\d+\s*)?\s*삭제\s*$")

# 항 삭제: "② 삭제" / "(2) 삭제" 같은 건 케이스가 있을 수 있어서 넓게
DEL_PAR_CIRCLED_PAT = re.compile(r"^\s*[①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳]\s*삭제\s*$")
DEL_PAR_PAREN_NUM_PAT = re.compile(r"^\s*\(?\s*\d+\s*\)?\s*삭제\s*$")  # 혹시 (2) 삭제 같은 경우

# 호 삭제: "1 삭제" (항 삭제와 겹칠 수 있으니 level로 구분해서 사용)
DEL_ITEM_NUM_PAT = re.compile(r"^\s*\d+\s*삭제\s*$")

# 목 삭제: "가 삭제" / "가. 삭제" / "가) 삭제" 등
DEL_SUBITEM_PAT = re.compile(r"^\s*[가-힣]\s*[\.\)]?\s*삭제\s*$")

def is_deleted_text_by_level(level: str, text: str) -> bool:
    """
    level별 '삭제' 전용 노드 판정.
    - article: '제n조 삭제'
    - paragraph: '② 삭제' (주로 원형숫자)
    - item: '1 삭제'
    - subitem: '가 삭제'
    """
    if not text:
        return False
    t = text.strip()

    if level == "article":
        return bool(DEL_ARTICLE_PAT.fullmatch(t))

    if level == "paragraph":
        # 항 삭제는 보통 원형숫자지만, 데이터에 따라 (2) 삭제 같은 게 섞일 수 있어 둘 다 허용
        return bool(DEL_PAR_CIRCLED_PAT.fullmatch(t) or DEL_PAR_PAREN_NUM_PAT.fullmatch(t))

    if level == "item":
        return bool(DEL_ITEM_NUM_PAT.fullmatch(t))

    if level == "subitem":
        return bool(DEL_SUBITEM_PAT.fullmatch(t))

    return False



# =========================================================
# 3) is_indexable 재부여
# =========================================================
def reassign_is_indexable(node: Dict[str, Any]) -> bool:
    level = node.get("level")
    text = (node.get("text") or "").strip()

    if is_deleted_text_by_level(level, text):
        return False
    
    if is_link_only_junyong(text):
        return False
    
    if level == "article":
        article_title = (node.get("article_title") or "").strip()

        # 0) '준용규정'인데 본문 없이 헤더만 있으면 False
        if is_junyong_header_only(article_title, text):
            return False

        # 1) 표제(제n절/장/관/편)는 False
        if is_heading_article_text(text):
            return False
    
        
        return True


    return True


# =========================================================
# 4) IO
# =========================================================
def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: str, rows: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [3]:

# ============================================================
#  X) 통합 실행 함수: XML → base JSONL → refs 추가 → is_indexable 재부여 → 최종 JSONL
#     ※ 두 노트북의 '실제 실행 순서'를 그대로 한 함수로 묶었습니다.
# ============================================================
import tempfile
from pathlib import Path

def apply_is_indexable_jsonl(input_path: str, output_path: str) -> int:
    """add_refs_and_is_indexable.ipynb에서 하던 방식 그대로:
    jsonl을 읽어서 각 row에 reassign_is_indexable(row)를 적용 후 다시 저장.
    """
    rows = read_jsonl(input_path)
    for r in rows:
        r["is_indexable"] = reassign_is_indexable(r)
    write_jsonl(output_path, rows)
    return len(rows)

def xml_to_jsonl_rag(
    xml_path: str,
    output_path: str,
    *,
    source: str = "law_xml",
    domain: str = "statute",
    include_article_level: bool = True,
    law_name_to_id: dict | None = None,
) -> int:
    if law_name_to_id is None:
        law_name_to_id = {}

    xml_path = str(xml_path)
    output_path = str(output_path)

    with tempfile.TemporaryDirectory() as td:
        td = Path(td)
        tmp_base = str(td / (Path(output_path).stem + ".__base__.jsonl"))
        tmp_refs = str(td / (Path(output_path).stem + ".__refs__.jsonl"))

        # 1) base jsonl (law_to_jsonl)
        n = xml_to_jsonl_rag_base(
            xml_path,
            tmp_base,
            source=source,
            domain=domain,
            include_article_level=include_article_level,
        )

        # 2) refs 추가 (add_refs)
        enrich_jsonl_with_refs(
            input_path=tmp_base,
            output_path=tmp_refs,
            law_name_to_id=law_name_to_id,
        )

        # 3) is_indexable 재부여 (add_refs의 reindex 단계)
        apply_is_indexable_jsonl(
            input_path=tmp_refs,
            output_path=output_path,
        )

    return n


### 적용 테스트

In [32]:
import os, json

# ===== 경로 설정 =====
NEED_LAWS_JSON = "../data/need_laws.json"     # 필요 법령 매핑(한글명 -> 파일코드)
RAW_DIR = "../data/law_rawdata"              # XML 폴더 (예: Civil_Law.xml)
OUT_DIR = "../data/law_jsonldata"            # 최종 JSONL 폴더 (예: Civil_Law_final.jsonl)
LAW_MAP_JSONL = "../data/law_map.jsonl"      # (선택) 외부 법령명→law_id 매핑

os.makedirs(OUT_DIR, exist_ok=True)

# ===== need_laws.json 로드 =====
with open(NEED_LAWS_JSON, "r", encoding="utf-8") as f:
    need_laws: dict = json.load(f)

# 파일코드 리스트 (예: ["Consumer_Basic_Law", "Civil_Law", ...])
law_codes = list(need_laws.values())

# (선택) law_map 로드 (없으면 빈 dict로 진행)
law_name_to_id = load_law_name_map(LAW_MAP_JSONL) if os.path.exists(LAW_MAP_JSONL) else {}

print(f"need_laws count: {len(need_laws)}")
print(f"law_codes: {law_codes}")

# ===== 일괄 변환 =====
results = []
for code in law_codes:
    xml_path = os.path.join(RAW_DIR, f"{code}.xml")
    out_path = os.path.join(OUT_DIR, f"{code}.jsonl")

    # Civil_Law.xml은 반드시 있다고 가정 (그 외는 없을 수도 있으니 체크)
    if not os.path.exists(xml_path):
        print(f"⚠️ SKIP (xml not found): {xml_path}")
        results.append((code, 0, "missing_xml"))
        continue

    n = xml_to_jsonl_rag(xml_path, out_path, law_name_to_id=law_name_to_id)
    print(f"✅ {code}: {n} rows -> {out_path}")
    results.append((code, n, "ok"))


need_laws count: 11
law_codes: ['Consumer_Basic_Law', 'E_Commerce_Consumer_Law', 'E_Transaction_Law', 'Content_Industry_Promotion_Law', 'Product_Liability_Law', 'Terms_Regulation_Law', 'Fair_Ads_Law', 'Installment_Sales_Law', 'Direct_Sales_Law', 'Civil_Law', 'Commercial_Law']
✅ 594 rows saved to C:\Users\Playdata\AppData\Local\Temp\tmp9imvwql5\Consumer_Basic_Law.__base__.jsonl
✅ Consumer_Basic_Law: 594 rows -> ../data/law_jsonldata\Consumer_Basic_Law.jsonl
✅ 357 rows saved to C:\Users\Playdata\AppData\Local\Temp\tmp9e2ov0kf\E_Commerce_Consumer_Law.__base__.jsonl
✅ E_Commerce_Consumer_Law: 357 rows -> ../data/law_jsonldata\E_Commerce_Consumer_Law.jsonl
✅ 398 rows saved to C:\Users\Playdata\AppData\Local\Temp\tmpey5zentx\E_Transaction_Law.__base__.jsonl
✅ E_Transaction_Law: 398 rows -> ../data/law_jsonldata\E_Transaction_Law.jsonl
✅ 311 rows saved to C:\Users\Playdata\AppData\Local\Temp\tmp1lrkj154\Content_Industry_Promotion_Law.__base__.jsonl
✅ Content_Industry_Promotion_Law: 311 rows -

In [33]:
# 결과 요약
ok = [r for r in results if r[2] == "ok"]
sk = [r for r in results if r[2] != "ok"]

print("\n=== SUMMARY ===")
print(f"OK: {len(ok)} / {len(results)}")
for code, n, _ in ok:
    print(f"- {code}: {n}")

if sk:
    print("\nSKIPPED:")
    for code, n, status in sk:
        print(f"- {code}: {status}")



=== SUMMARY ===
OK: 11 / 11
- Consumer_Basic_Law: 594
- E_Commerce_Consumer_Law: 357
- E_Transaction_Law: 398
- Content_Industry_Promotion_Law: 311
- Product_Liability_Law: 40
- Terms_Regulation_Law: 199
- Fair_Ads_Law: 142
- Installment_Sales_Law: 462
- Direct_Sales_Law: 570
- Civil_Law: 2547
- Commercial_Law: 3653
